# Train YOLOv8n-seg (Kaggle or Colab)

Works for both models - upload ONE zip:
- `yolo_ds.zip` -> dish model -> save best.pt as `models/best.pt`
- `food_ds.zip` -> leftover model -> save best.pt as `models/food_best.pt`

**Kaggle:** Settings -> *Accelerator: GPU T4 x2* and *Internet: On*. Upload the zip with **+ Add Input -> Upload -> New dataset**.

**Colab:** Runtime -> Change runtime type -> *T4 GPU*. Upload the zip with the folder panel.

Run all cells (about 15-25 min), then download **best.pt** and **training_results.zip**.

In [ ]:
!pip install -q ultralytics
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# find the uploaded dataset (yolo_ds = dish model, food_ds = leftover model)
import glob, os, shutil, zipfile
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
os.chdir(WORK)
DS = os.path.join(WORK, "ds")
if not os.path.exists(DS):
    yamls = [y for y in glob.glob("/kaggle/input/**/data.yaml", recursive=True)]
    zips = glob.glob("/kaggle/input/**/*_ds.zip", recursive=True) + glob.glob(f"{WORK}/*_ds.zip")
    if yamls:                                   # Kaggle unzipped it (read-only) -> copy
        shutil.copytree(os.path.dirname(yamls[0]), DS)
    elif zips:
        zipfile.ZipFile(zips[0]).extractall("unz")
        shutil.copytree(os.path.dirname(glob.glob("unz/**/data.yaml", recursive=True)[0]), DS)
    else:
        raise SystemExit("dataset not found - upload yolo_ds.zip or food_ds.zip first")
y = open(f"{DS}/data.yaml").read()
y = "\n".join(l for l in y.splitlines() if not l.startswith("path:"))
open(f"{DS}/data.yaml", "w").write(f"path: {DS}\n" + y + "\n")
print(open(f"{DS}/data.yaml").read())
print("train images:", len(os.listdir(f"{DS}/images/train")), " val images:", len(os.listdir(f"{DS}/images/val")))

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n-seg.pt")          # pretrained on COCO, we fine-tune it
model.train(data=f"{DS}/data.yaml", epochs=80, imgsz=640, batch=16,
            patience=25, seed=0, project="runs_seg", name="food", exist_ok=True,
            degrees=30, fliplr=0.5, flipud=0.5, hsv_v=0.4)
SAVE = str(model.trainer.save_dir)      # where this Ultralytics version saved the run
print("run saved in", SAVE)

In [ ]:
# validation scores (mask mAP) + copy weights to an easy place
m = YOLO(f"{SAVE}/weights/best.pt")
r = m.val(data=f"{DS}/data.yaml", imgsz=640, verbose=False)
print("mask mAP50:", round(r.seg.map50, 3), "  mask mAP50-95:", round(r.seg.map, 3))
shutil.copy(f"{SAVE}/weights/best.pt", "best.pt")
shutil.make_archive("training_results", "zip", SAVE)   # curves + confusion matrix for the report
print("Download best.pt (and training_results.zip for the report figures)")

In [ ]:
# Colab only: download directly
try:
    from google.colab import files
    files.download("best.pt"); files.download("training_results.zip")
except ImportError:
    print("On Kaggle: open the Output panel (right side, /kaggle/working) and download best.pt + training_results.zip")